# STaR/ReST^EM: Iterative Self-Improvement

This notebook implements **STaR** (Self-Taught Reasoner) / **ReST^EM** (Reinforced Self-Training
with Expectation-Maximization) for iterative self-improvement of the GSPO-trained model.

**Training pipeline stage:** 4 of 4 (SFT -> SimPO -> GSPO -> **STaR**)

**Algorithm:**
1. Generate multiple completions per problem (N=16)
2. Verify correctness using hybrid verifier (symbolic + heuristic)
3. Keep only correct completions as training data
4. SFT for 1 epoch on the filtered correct completions (domain-balanced)
5. Repeat for 2-3 iterations, tracking per-domain accuracy
6. Stop if improvement < `min_improvement` threshold

**Final evaluation:** STEM-30 (30 curated problems) + MATH-100 (100 problems from MATH benchmark)

**Domains:** Mathematics, Physics, Chemistry, Biology, Computer Science

In [ ]:
# Install dependencies
!pip install -q unsloth trl peft transformers datasets
!pip install -q accelerate bitsandbytes sentencepiece protobuf
!pip install -q sympy chempy  # For verification

In [ ]:
# ============================================================
# Configuration
# ============================================================

# Model
BASE_MODEL = "unsloth/Qwen3-4B"
GSPO_CHECKPOINT = "/content/drive/MyDrive/MITS/checkpoints/gspo_qwen3_4b/final_adapter"
OUTPUT_DIR = "/content/drive/MyDrive/MITS/checkpoints/star_qwen3_4b"

# STaR hyperparameters
ITERATIONS = 3              # Maximum STaR iterations
COMPLETIONS_PER_PROBLEM = 16  # Number of completions to generate per problem
MIN_IMPROVEMENT = 0.02      # Stop if accuracy improvement is below this threshold

# Generation parameters
MAX_SEQ = 2048
MAX_NEW_TOKENS = 1024
TEMPERATURE = 0.8           # Higher temperature for diversity in generation
TOP_P = 0.95

# SFT parameters for each iteration
SFT_EPOCHS = 1
SFT_BATCH_SIZE = 4
SFT_GRADIENT_ACCUMULATION = 4
SFT_LEARNING_RATE = 1e-5    # Lower LR for refinement
SFT_WARMUP_STEPS = 20

# Paths
GSPO_PROBLEMS_PATH = "training/data/gspo_problems.jsonl"
STEM30_PATH = "evaluation/benchmarks/stem_30.jsonl"
MATH100_PATH = "evaluation/benchmarks/math_100.jsonl"

# Domains
DOMAINS = ["math", "physics", "chemistry", "biology", "cs"]

print(f"Base model: {BASE_MODEL}")
print(f"GSPO checkpoint: {GSPO_CHECKPOINT}")
print(f"STaR iterations: {ITERATIONS}, completions/problem: {COMPLETIONS_PER_PROBLEM}")
print(f"Min improvement threshold: {MIN_IMPROVEMENT}")

In [ ]:
# ============================================================
# Mount Google Drive (optional) and load GSPO checkpoint + problems from HF
# ============================================================
import json
import os
import sys
import random
import math
from collections import Counter, defaultdict

# Try mounting Drive; fall back to local
DRIVE_MOUNTED = False
try:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE_MOUNTED = True
    print("Google Drive mounted successfully")
except Exception as e:
    print(f"Drive mount failed ({e}), using local storage")
    OUTPUT_DIR = "/content/checkpoints/star_qwen3_4b"
    GSPO_CHECKPOINT = "/content/checkpoints/gspo_qwen3_4b/final_adapter"

# Verify GSPO checkpoint
assert os.path.exists(GSPO_CHECKPOINT), f"GSPO checkpoint not found at {GSPO_CHECKPOINT}"
print(f"GSPO checkpoint verified: {GSPO_CHECKPOINT}")

# Add project root to path
if DRIVE_MOUNTED:
    project_root = "/content/drive/MyDrive/MITS"
    if project_root not in sys.path:
        sys.path.insert(0, project_root)

os.makedirs(OUTPUT_DIR, exist_ok=True)

# Load model
import torch
from unsloth import FastLanguageModel
from peft import PeftModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE_MODEL,
    max_seq_length=MAX_SEQ,
    load_in_4bit=True,
    dtype=None,
)

model = PeftModel.from_pretrained(model, GSPO_CHECKPOINT)
model = model.merge_and_unload()
print(f"Loaded GSPO checkpoint from {GSPO_CHECKPOINT}")

# Apply fresh LoRA for STaR fine-tuning
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules="all-linear",
    use_gradient_checkpointing="unsloth",
    random_state=42,
)

trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
total_params = sum(p.numel() for p in model.parameters())
print(f"Trainable: {trainable_params:,} / {total_params:,}")

# Load problems from HuggingFace
from datasets import load_dataset

print("Loading GSPO problems from Siesher/mits-stem-training-data...")
hf_ds = load_dataset("Siesher/mits-stem-training-data", "gspo")
problems = [dict(r) for r in hf_ds["train"]] + [dict(r) for r in hf_ds["test"]]

print(f"Loaded {len(problems)} problems")
domain_counts = Counter(p.get("domain", "unknown") for p in problems)
for d, c in sorted(domain_counts.items()):
    print(f"  {d}: {c}")

In [ ]:
# ============================================================
# STaR iteration function
# ============================================================
from trl import SFTTrainer
from transformers import TrainingArguments
from datasets import Dataset

# Import hybrid verifier
try:
    from training.scripts.verify_answers import verify_answer
    print("Imported verify_answer from training.scripts.verify_answers")
except ImportError:
    print("WARNING: Could not import verify_answers, using fallback verifier")

    import sympy
    import re

    def verify_answer(completion, gold_answer, domain="math"):
        """Hybrid verifier: symbolic check + heuristic fallback.

        Returns True if the completion contains the correct answer.
        """
        # Extract boxed answer
        boxed = re.findall(r"\\boxed\{([^}]+)\}", completion)
        extracted = boxed[-1].strip() if boxed else ""

        if not extracted:
            # Fallback: last number in response
            numbers = re.findall(r"[-+]?\d*\.?\d+", completion)
            extracted = numbers[-1] if numbers else ""

        if not extracted:
            return False

        # Symbolic equivalence check (math, physics)
        if domain in ("math", "physics"):
            try:
                pred = sympy.sympify(extracted)
                gold = sympy.sympify(str(gold_answer))
                if sympy.simplify(pred - gold) == 0:
                    return True
            except (sympy.SympifyError, TypeError, ValueError):
                pass

            # Numeric tolerance check
            try:
                pred_num = float(extracted)
                gold_num = float(gold_answer)
                if abs(pred_num - gold_num) / max(abs(gold_num), 1e-10) < 0.05:
                    return True
            except (ValueError, TypeError):
                pass

        # String match (all domains)
        if extracted.strip().lower() == str(gold_answer).strip().lower():
            return True

        # Keyword presence for conceptual domains
        if domain in ("biology", "cs", "chemistry"):
            if str(gold_answer).strip().lower() in completion.lower():
                return True

        return False


def star_iteration(model, tokenizer, problems, iteration_num, output_dir):
    """Run one STaR iteration:
    1. Generate N completions per problem
    2. Verify correctness
    3. Keep correct completions
    4. SFT for 1 epoch on correct completions (domain-balanced)

    Returns per-domain accuracy dict.
    """
    print(f"\n{'='*60}")
    print(f"STaR Iteration {iteration_num}")
    print(f"{'='*60}")

    # Step 1: Generate completions
    FastLanguageModel.for_inference(model)

    correct_examples = []  # (messages_text, domain)
    domain_stats = defaultdict(lambda: {"total": 0, "correct": 0, "completions_correct": 0, "completions_total": 0})

    print(f"Generating {COMPLETIONS_PER_PROBLEM} completions for {len(problems)} problems...")

    for i, problem in enumerate(problems):
        domain = problem.get("domain", "math")
        answer = problem.get("answer", "")
        domain_stats[domain]["total"] += 1

        messages = [
            {"role": "system", "content": "You are a Socratic STEM tutor. Solve step by step and put your final answer in \\boxed{}."},
            {"role": "user", "content": problem["prompt"]},
        ]
        prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=MAX_SEQ // 2).to(model.device)

        # Generate multiple completions
        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=MAX_NEW_TOKENS,
                temperature=TEMPERATURE,
                top_p=TOP_P,
                do_sample=True,
                num_return_sequences=COMPLETIONS_PER_PROBLEM,
            )

        # Verify each completion
        problem_has_correct = False
        for j in range(COMPLETIONS_PER_PROBLEM):
            completion = tokenizer.decode(
                outputs[j][inputs["input_ids"].shape[1]:],
                skip_special_tokens=True,
            )
            domain_stats[domain]["completions_total"] += 1

            if verify_answer(completion, answer, domain):
                domain_stats[domain]["completions_correct"] += 1
                problem_has_correct = True

                # Format as training example
                full_messages = messages + [{"role": "assistant", "content": completion}]
                text = tokenizer.apply_chat_template(
                    full_messages,
                    tokenize=False,
                    add_generation_prompt=False,
                )
                correct_examples.append({"text": text, "domain": domain})

        if problem_has_correct:
            domain_stats[domain]["correct"] += 1

        if (i + 1) % 20 == 0:
            print(f"  Processed {i+1}/{len(problems)} problems, {len(correct_examples)} correct completions so far")

    # Print generation stats
    print(f"\nGeneration results:")
    domain_accuracy = {}
    for domain in DOMAINS:
        s = domain_stats[domain]
        if s["total"] > 0:
            prob_acc = s["correct"] / s["total"]
            comp_acc = s["completions_correct"] / max(s["completions_total"], 1)
            domain_accuracy[domain] = prob_acc
            print(f"  {domain}: {prob_acc:.1%} problems solved ({s['correct']}/{s['total']}), "
                  f"{comp_acc:.1%} completions correct ({s['completions_correct']}/{s['completions_total']})")

    print(f"\nTotal correct training examples: {len(correct_examples)}")

    if len(correct_examples) == 0:
        print("WARNING: No correct completions generated. Skipping SFT.")
        return domain_accuracy

    # Step 2: Domain-balanced SFT on correct completions
    print("\nRunning domain-balanced SFT on correct completions...")

    # Balance domains
    domain_examples = defaultdict(list)
    for ex in correct_examples:
        domain_examples[ex["domain"]].append(ex)

    max_domain_size = max(len(v) for v in domain_examples.values()) if domain_examples else 0
    balanced_examples = []
    for domain in DOMAINS:
        exs = domain_examples.get(domain, [])
        if exs:
            multiplied = exs * (max_domain_size // len(exs) + 1)
            balanced_examples.extend(multiplied[:max_domain_size])

    random.shuffle(balanced_examples)
    print(f"  Balanced training set: {len(balanced_examples)} examples")

    train_ds = Dataset.from_list([{"text": ex["text"]} for ex in balanced_examples])

    FastLanguageModel.for_training(model)

    iter_output = os.path.join(output_dir, f"iteration_{iteration_num}")
    os.makedirs(iter_output, exist_ok=True)

    training_args = TrainingArguments(
        output_dir=iter_output,
        num_train_epochs=SFT_EPOCHS,
        per_device_train_batch_size=SFT_BATCH_SIZE,
        gradient_accumulation_steps=SFT_GRADIENT_ACCUMULATION,
        learning_rate=SFT_LEARNING_RATE,
        lr_scheduler_type="cosine",
        warmup_steps=SFT_WARMUP_STEPS,
        weight_decay=0.01,
        logging_steps=10,
        save_steps=200,
        save_total_limit=1,
        bf16=torch.cuda.is_bf16_supported(),
        fp16=not torch.cuda.is_bf16_supported(),
        optim="adamw_8bit",
        seed=42 + iteration_num,
        report_to="none",
        remove_unused_columns=False,
    )

    trainer = SFTTrainer(
        model=model,
        tokenizer=tokenizer,
        train_dataset=train_ds,
        max_seq_length=MAX_SEQ,
        dataset_text_field="text",
        packing=True,
        args=training_args,
    )

    result = trainer.train()
    print(f"  SFT loss: {result.training_loss:.4f}")

    # Save iteration checkpoint
    trainer.save_model(os.path.join(iter_output, "adapter"))
    print(f"  Saved to {iter_output}")

    return domain_accuracy

In [ ]:
# ============================================================
# Main loop: run 2-3 iterations, track per-domain accuracy
# ============================================================

iteration_results = []
prev_overall_accuracy = 0.0

for iteration in range(1, ITERATIONS + 1):
    domain_accuracy = star_iteration(
        model=model,
        tokenizer=tokenizer,
        problems=problems,
        iteration_num=iteration,
        output_dir=OUTPUT_DIR,
    )

    # Calculate overall accuracy
    accuracies = [v for v in domain_accuracy.values() if isinstance(v, (int, float))]
    overall_accuracy = sum(accuracies) / len(accuracies) if accuracies else 0.0

    iteration_results.append({
        "iteration": iteration,
        "domain_accuracy": domain_accuracy,
        "overall_accuracy": overall_accuracy,
    })

    print(f"\nIteration {iteration} overall accuracy: {overall_accuracy:.1%}")

    # Check improvement threshold
    improvement = overall_accuracy - prev_overall_accuracy
    print(f"Improvement over previous: {improvement:+.4f}")

    if iteration > 1 and improvement < MIN_IMPROVEMENT:
        print(f"\nStopping early: improvement ({improvement:.4f}) < threshold ({MIN_IMPROVEMENT})")
        break

    prev_overall_accuracy = overall_accuracy

# Summary
print("\n" + "="*60)
print("STaR Training Summary")
print("="*60)
for res in iteration_results:
    print(f"  Iteration {res['iteration']}: overall={res['overall_accuracy']:.1%}")
    for domain, acc in res["domain_accuracy"].items():
        print(f"    {domain}: {acc:.1%}")

# Save iteration logs
logs_path = os.path.join(OUTPUT_DIR, "star_iteration_logs.json")
with open(logs_path, "w") as f:
    json.dump(iteration_results, f, indent=2, default=str)
print(f"\nIteration logs saved to {logs_path}")

In [ ]:
# ============================================================
# Final evaluation: STEM-30 + MATH-100
# ============================================================

FastLanguageModel.for_inference(model)


def evaluate_benchmark(benchmark_path, benchmark_name):
    """Evaluate model on a benchmark dataset."""
    drive_path = f"/content/drive/MyDrive/MITS/{benchmark_path}"
    local_path = benchmark_path

    eval_path = drive_path if os.path.exists(drive_path) else local_path
    if not os.path.exists(eval_path):
        print(f"WARNING: Benchmark {benchmark_name} not found at {eval_path}, skipping")
        return None

    # Load benchmark
    bench_problems = []
    with open(eval_path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                bench_problems.append(json.loads(line))

    print(f"\n--- {benchmark_name} ({len(bench_problems)} problems) ---")

    domain_results = defaultdict(lambda: {"correct": 0, "total": 0})

    for i, problem in enumerate(bench_problems):
        domain = problem.get("domain", "math")
        answer = problem.get("answer", "")

        messages = [
            {"role": "system", "content": "You are a Socratic STEM tutor. Solve step by step and put your final answer in \\boxed{}."},
            {"role": "user", "content": problem["prompt"]},
        ]
        prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=MAX_SEQ // 2).to(model.device)

        # Best-of-N: generate 4 completions, take majority vote
        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=MAX_NEW_TOKENS,
                temperature=0.6,
                top_p=0.9,
                do_sample=True,
                num_return_sequences=4,
            )

        correct_count = 0
        for j in range(4):
            completion = tokenizer.decode(
                outputs[j][inputs["input_ids"].shape[1]:],
                skip_special_tokens=True,
            )
            if verify_answer(completion, answer, domain):
                correct_count += 1

        domain_results[domain]["total"] += 1
        if correct_count >= 2:  # Majority (2+ out of 4)
            domain_results[domain]["correct"] += 1

        if (i + 1) % 10 == 0:
            total_correct = sum(d["correct"] for d in domain_results.values())
            total_total = sum(d["total"] for d in domain_results.values())
            print(f"  {i+1}/{len(bench_problems)}: running accuracy = {total_correct}/{total_total} ({100*total_correct/max(total_total,1):.1f}%)")

    # Print results
    print(f"\n{benchmark_name} Results:")
    total_c, total_t = 0, 0
    results_dict = {}
    for domain in DOMAINS + ["unknown"]:
        m = domain_results[domain]
        if m["total"] > 0:
            acc = 100 * m["correct"] / m["total"]
            print(f"  {domain}: {acc:.1f}% ({m['correct']}/{m['total']})")
            results_dict[domain] = {"accuracy": acc, "correct": m["correct"], "total": m["total"]}
            total_c += m["correct"]
            total_t += m["total"]

    overall_acc = 100 * total_c / max(total_t, 1)
    print(f"  OVERALL: {overall_acc:.1f}% ({total_c}/{total_t})")
    results_dict["overall"] = {"accuracy": overall_acc, "correct": total_c, "total": total_t}

    return results_dict


# Run evaluations
stem30_results = evaluate_benchmark(STEM30_PATH, "STEM-30")
math100_results = evaluate_benchmark(MATH100_PATH, "MATH-100")

# Save evaluation results
eval_results = {
    "stem_30": stem30_results,
    "math_100": math100_results,
    "star_iterations": len(iteration_results),
    "iteration_results": iteration_results,
}

eval_path = os.path.join(OUTPUT_DIR, "final_eval_results.json")
with open(eval_path, "w") as f:
    json.dump(eval_results, f, indent=2, default=str)
print(f"\nFull evaluation results saved to {eval_path}")

In [ ]:
# ============================================================
# Save final adapter
# ============================================================

final_adapter_path = os.path.join(OUTPUT_DIR, "final_adapter")
model.save_pretrained(final_adapter_path)
tokenizer.save_pretrained(final_adapter_path)
print(f"Final STaR adapter saved to {final_adapter_path}")

# Save training config
config_to_save = {
    "stage": "star",
    "base_model": BASE_MODEL,
    "gspo_checkpoint": GSPO_CHECKPOINT,
    "iterations_completed": len(iteration_results),
    "max_iterations": ITERATIONS,
    "completions_per_problem": COMPLETIONS_PER_PROBLEM,
    "min_improvement": MIN_IMPROVEMENT,
    "sft_learning_rate": SFT_LEARNING_RATE,
    "sft_epochs_per_iteration": SFT_EPOCHS,
    "temperature": TEMPERATURE,
    "total_problems": len(problems),
    "final_overall_accuracy": iteration_results[-1]["overall_accuracy"] if iteration_results else None,
    "stem30_overall": stem30_results.get("overall", {}).get("accuracy") if stem30_results else None,
    "math100_overall": math100_results.get("overall", {}).get("accuracy") if math100_results else None,
}
config_path = os.path.join(OUTPUT_DIR, "training_config.json")
with open(config_path, "w") as f:
    json.dump(config_to_save, f, indent=2)
print(f"Config saved to {config_path}")

print("\n" + "="*60)
print("TRAINING PIPELINE COMPLETE")
print("="*60)
print(f"Final model: {final_adapter_path}")
print(f"Pipeline: SFT -> SimPO -> GSPO -> STaR ({len(iteration_results)} iterations)")
if stem30_results and "overall" in stem30_results:
    print(f"STEM-30 accuracy: {stem30_results['overall']['accuracy']:.1f}%")
if math100_results and "overall" in math100_results:
    print(f"MATH-100 accuracy: {math100_results['overall']['accuracy']:.1f}%")
print("\nThe final adapter can be exported to GGUF for Ollama deployment.")